In [0]:
with open("/dbfs/databricks-datasets/tpch/README.md", "r") as f:
    content = f.read()
    displayHTML(f"<pre>{content}</pre>")

In [0]:
spark.sql("SHOW TABLES IN samples.tpch").show()

In [0]:
df_lineitem = spark.table("samples.tpch.lineitem")
df_orders = spark.table("samples.tpch.orders")

In [0]:
print(f"Number of rows in lineitem: {df_lineitem.count():,}")
print(f"Number of rows in orders: {df_orders.count():,}")

In [0]:
df_lineitem.limit(10).display()

In [0]:
df_lineitem.printSchema()

In [0]:
df_orders.printSchema()

In [0]:
from pyspark.sql import functions as F

df_lineitem.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_lineitem.columns
]).display()

In [0]:
display(
    df_lineitem.groupBy("l_shipmode")
    .agg(
        F.count("*").alias("count"),
        (F.count("*") / df_lineitem.count() * 100).alias("share_pct")
    )
    .orderBy(F.desc("count"))
)

In [0]:
display(df_lineitem.groupBy("l_linestatus").count())

In [0]:
display(df_lineitem.groupBy("l_returnflag").count())

In [0]:
df_lineitem.select(
    F.min("l_shipdate").alias("min_ship"),
    F.max("l_shipdate").alias("max_ship"),
    F.min("l_commitdate").alias("min_commit"),
    F.max("l_commitdate").alias("max_commit"),
    F.min("l_receiptdate").alias("min_receipt"),
    F.max("l_receiptdate").alias("max_receipt")
).display()

A shipment can’t leave before the order was placed (shipdate < orderdate):

In [0]:
df_joined_dates = df_lineitem.join(
    df_orders,
    df_lineitem.l_orderkey == df_orders.o_orderkey,
    "inner"
)

In [0]:
invalid_order_dates = df_joined_dates.filter(F.col("l_shipdate") < F.col("o_orderdate"))
print(f"Anomalies (shipdate < orderdate): {invalid_order_dates.count()}")

A shipment can’t be received before it was shipped (receiptdate < shipdate):

In [0]:
invalid_transit_dates = df_lineitem.filter(F.col("l_receiptdate") < F.col("l_shipdate"))
print(f"Anomalies (receiptdate < shipdate): {invalid_transit_dates.count()}")

In [0]:
orphaned_lineitems = df_lineitem.join(
    df_orders,
    df_lineitem.l_orderkey == df_orders.o_orderkey,
    "left_anti"
)
print(f"Lineitems without orders: {orphaned_lineitems.count()}")

In [0]:
orders_without_items = df_orders.join(
    df_lineitem,
    df_orders.o_orderkey == df_lineitem.l_orderkey,
    "left_anti"
)
print(f"Orders without lineitems: {orders_without_items.count()}")

In [0]:
items_per_order = df_lineitem.groupBy("l_orderkey").agg(F.count("*").alias("items_cnt"))
display(items_per_order.describe("items_cnt"))

In [0]:
df_transit = df_lineitem.withColumn(
    "transit_days",
    F.datediff(F.col("l_receiptdate"), F.col("l_shipdate"))
)

In [0]:
df_transit.select(
    F.min("transit_days").alias("min_transit_days"),
    F.max("transit_days").alias("max_transit_days"),
    F.avg("transit_days").alias("avg_transit_days")
).display()

In [0]:
total_items = df_lineitem.count()
delayed_items = df_lineitem.filter(F.col("l_receiptdate") > F.col("l_commitdate")).count()

print(f"Delay rate: {(delayed_items / total_items) * 100:.2f}%")